In [0]:
# Pipeline Mode Selection Widget

dbutils.widgets.dropdown("mode", "train", ["train", "predict"], "Pipeline Mode")
mode = dbutils.widgets.get("mode")
suffix = "_train" if mode == "train" else "_predict"

In [0]:
# Set mode at the top ('train' or 'predict')
try:
    mode = dbutils.widgets.get("mode")
except:
    mode = "train"  # Fallback; set as needed

suffix = "_train" if mode == "train" else "_predict"

silver_table_name = f"marchmadness.silver_kenpom{suffix}"

if mode == "train":
    kenpom_path = f"/Volumes/workspace/marchmadnesslandingzone/kenpom{suffix}/*.csv"
else:
    kenpom_path = "/Volumes/workspace/marchmadnesslandingzone/predict/KenPomDataRaw_Predict.csv"

print(suffix)
print(kenpom_path)


In [0]:
from pyspark.sql.functions import regexp_extract, regexp_replace, trim, col, split, concat
from pyspark.sql import functions as F

# Load all CSVs from the kenpom directory
# Use mode-based kenpom_path
kenpom_path = kenpom_path

df = spark.read.option("header", True).csv(kenpom_path)

display(df)

In [0]:
# Extract the seed (last number following space at end of Team string)
df = df.withColumn("Seed", regexp_extract(col("Team"), r" (\d+)$", 1))

# Only keep rows with extracted Seed
kenpom_seeded = df.filter(col("Seed") != "")

# Clean Team name: remove trailing numbers, spaces, and periods
kenpom_seeded = kenpom_seeded.withColumn(
    "Team",
    trim(regexp_replace(regexp_replace(col("Team"), r"(\d+)$", ""), r"[.]", ""))
)

# Split Record column to get Wins
kenpom_seeded = kenpom_seeded.withColumn(
    "Wins", split(col("Record"), "-").getItem(0)
)

# Select the major columns needed
target_cols = [
     "Team", "Year", "Seed", "Wins",
    "AdjEM", "AdjO", "AdjD", "SoS_AdjEM", "OppO", "OppD"
]
df_target = kenpom_seeded.select(*target_cols)

display(df_target)


In [0]:
from pyspark.sql.functions import lower, regexp_replace

# Define kenpom_lower before the join (minimal fix for NameError)
kenpom_lower = df_target.withColumn(
    "Team_clean",
    lower(regexp_replace(regexp_replace(col("Team"), r"[.]", ""), r"[-]", " "))
)

# Load Kaggle MTeamsSpellings.csv with alternate names.  This is always in the training path.
spellings_df = spark.read.option("header", True).csv(f"/Volumes/workspace/marchmadnesslandingzone/kaggle_train/MTeamSpellings.csv")

# Remove periods, hyphens and lower-case team name columns for join
spellings_df = spellings_df.withColumn(
    "TeamName_clean",
    lower(regexp_replace(regexp_replace(col("TeamNameSpelling"), r"[.]", ""), r"[-]", " "))
)

# Join KenPom features with spelling alternatives
joined = kenpom_lower.join(spellings_df, kenpom_lower["Team_clean"] == spellings_df["TeamName_clean"], "left")\
            .drop('Team_clean', 'TeamNameSpelling', 'TeamName_clean').dropDuplicates()

# Identify teams with missing Team_Id
no_id_teams = joined.filter(col("TeamID").isNull()).select("Team")\
    .drop('TeamNameSpelling', 'TeamName_clean')

# Note - manually added the following to the file in 2026.
# Cal St Bakersfield
# Mississippi Valley St
# Saint Francis
# Southeast Missouri St
# Texas A&M Corpus Chris

# Output results
print(f"Teams missing Team_Id: {no_id_teams.count()}")
display(no_id_teams)

display(joined)

In [0]:
joined_key_df = joined.withColumn('Season_Team', concat(col('Year'), F.lit('_'), col('TeamID')))

display(joined_key_df)


In [0]:
from pyspark.sql.types import IntegerType, DoubleType

# Identify columns to convert (exclude 'Team' and 'Season_Team')
cols_to_convert = [col for col in joined_key_df.columns if col not in ['Team', 'Season_Team']]

# Define desired types for each column
type_map = {
    'Year': IntegerType(),
    'Seed': IntegerType(),
    'Wins': IntegerType(),
    'AdjEM': DoubleType(),
    'AdjO': DoubleType(),
    'AdjD': DoubleType(),
    'SoS_AdjEM': DoubleType(),
    'OppO': DoubleType(),
    'OppD': DoubleType(),
    'TeamID': IntegerType()
}

# Apply type conversions
for c in cols_to_convert:
    if c in type_map:
        joined_key_df = joined_key_df.withColumn(c, joined_key_df[c].cast(type_map[c]))

display(joined_key_df)

In [0]:
# Create schema if not exists
spark.sql("CREATE SCHEMA IF NOT EXISTS marchmadness")

# Save the transformed KenPom data to the mode-specific silver table
joined_key_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(silver_table_name)

# Verify a sample of the saved table
display(spark.table(silver_table_name))